In [ ]:
# Tek parça script: GPU kontrolü, clone, veri hazırlama, eğitim

import os, subprocess, sys, numpy as np, pickle
import torch # Import torch here

def run(cmd):
    print(f"\nRUN -> {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("ERROR:", result.stderr)
        sys.exit(1)

# GPU kontrol
if not torch.cuda.is_available():
    print("No GPU! Switch runtime to GPU and rerun."); sys.exit(1)
print("GPU OK:", torch.cuda.get_device_name(0))

# Dosya kontrol
if not os.path.isfile("/content/veri-seti.txt"):
    print("veri-seti.txt bulunamadı, upload et ve yeniden çalıştır."); sys.exit(1)
print("veri-seti.txt bulundu.")

# Repo clone & setup
if not os.path.isdir("/content/nanoGPT"):
    run("git clone https://github.com/karpathy/nanoGPT.git /content/nanoGPT")
os.chdir("/content/nanoGPT")

# Paketler
run("pip install --upgrade pip")
run("pip install torch tqdm numpy")

# Veri hazırlama (karakter tabanlı)
os.makedirs("data/custom", exist_ok=True)
run("cp /content/veri-seti.txt data/custom/input.txt")
print("Veri hazırlanıyor…")
data = open("data/custom/input.txt","r",encoding="utf-8").read()
chars = sorted(list(set(data))) # Convert set to list before sorting
stoi = {c:i for i,c in enumerate(chars)}
encoded = np.array([stoi[c] for c in data], dtype=np.uint16)
n = len(encoded)
np.array(encoded[:int(n*0.9)]).tofile("data/custom/train.bin")
np.array(encoded[int(n*0.9):]).tofile("data/custom/val.bin")
pickle.dump({'vocab_size':len(chars),'itos':{i:c for i,c in enumerate(chars)},
             'stoi':stoi}, open("data/custom/meta.pkl","wb"))
print("Veri .bin ve meta dosyaları oluşturuldu.")

# Config yaz
cfg = """out_dir = 'out-custom'
eval_interval = 50
log_interval = 10
eval_iters = 20
always_save_checkpoint = True
wandb_log = False
dataset = 'custom'
batch_size = 8
block_size = 128
n_layer = 4
n_head = 4
n_embd = 256
max_iters = 1000
lr_decay_iters = 1000
dropout = 0.1
compile = False # Disable torch.compile
"""
os.makedirs("config", exist_ok=True)
with open("config/train_custom.py","w") as f:
    f.write(cfg)

# Eğitim
run("python train.py config/train_custom.py --device=cuda")

GPU OK: Tesla T4
veri-seti.txt bulundu.

RUN -> git clone https://github.com/karpathy/nanoGPT.git /content/nanoGPT


RUN -> pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


RUN -> pip install torch tqdm numpy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install tiktoken

In [ ]:
import os, subprocess, sys

def run(cmd):
    print(f"\nRUN -> {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("ERROR:", result.stderr)
        sys.exit(1)

# nanoGPT dizinine git
os.chdir("/content/nanoGPT")

# Modeli kullanarak metin üret
# --out_dir: modelin kaydedildiği dizin
# --device: hangi cihazda çalıştırılacak (cuda = GPU)
# --num_samples: kaç farklı metin üretilecek
# --max_new_tokens: her bir metinde kaç yeni token (karakter) üretilecek
# --start: Metnin başlangıcı için bir prompt (isteğe bağlı).
#          Kendi metinlerinden bir başlangıç verebilirsin, yoksa rastgele başlar.
print("\n--- Modelden Metin Üretiliyor ---")
run("python sample.py --out_dir=out-custom --device=cuda --num_samples=2 --max_new_tokens=500 --start=\"Of \"")

FileNotFoundError: [Errno 2] No such file or directory: '/content/nanoGPT'

In [ ]:
import tiktoken, pickle, numpy as np, os, shutil, subprocess

enc = tiktoken.get_encoding("gpt2")
data = open("/content/veri-seti.txt", encoding="utf-8").read()
tokens = enc.encode_ordinary(data)
n = len(tokens)
train = np.array(tokens[:int(n*0.9)], dtype=np.uint16)
val  = np.array(tokens[int(n*0.9):], dtype=np.uint16)
train.tofile("data/custom/train.bin")
val.tofile("data/custom/val.bin")
meta = {"vocab_size": enc.n_vocab}
pickle.dump(meta, open("data/custom/meta.pkl", "wb"))

In [ ]:

# 3) Eğitimi başlat
!python /content/train_custom.py train

# 4) Eğitim bittikten sonra örnek al
!python /content/train_custom.py sample

  File "/content/train_custom.py", line 91
    self.get_batch(split, config, data_root):
                                            ^
SyntaxError: invalid syntax
  File "/content/train_custom.py", line 91
    self.get_batch(split, config, data_root):
                                            ^
SyntaxError: invalid syntax


In [ ]:
import os, subprocess, sys
os.makedirs("data/custom", exist_ok=True)

In [ ]:


%%writefile /content/train_custom.py
# ================== train_custom.py ==================
# Colab tek hücrede çalıştır:  !python train_custom.py
# Gerekli:  pip install torch tiktoken tqdm numpy

import os, math, time, pickle, argparse
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from tqdm import tqdm

# ---------- 1. Hyper-parametreler ----------
@torch.no_grad()
def default_config():
    return dict(
        device = "cuda" if torch.cuda.is_available() else "cpu",
        data_dir = "data/custom",
        out_dir = "out-custom",
        # model
        n_layer = 8,
        n_head = 8,
        n_embd = 512,
        block_size = 256,
        dropout = 0.1,
        bias = False,
        # training
        batch_size = 8,
        max_iters = 5000,
        learning_rate = 3e-4,
        beta1 = 0.9,
        beta2 = 0.99,
        weight_decay = 1e-1,
        grad_clip = 1.0,
        lr_decay_iters = 5000,
        min_lr = 3e-5,
        warmup_iters = 100,
        eval_interval = 100,
        eval_iters = 50,
        save_interval = 1000,
    )

# ---------- 2. GPT Model ----------
class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config["n_embd"] % config["n_head"] == 0
        self.c_attn = nn.Linear(config["n_embd"], 3*config["n_embd"], bias=config["bias"])
        self.c_proj = nn.Linear(config["n_embd"], config["n_embd"], bias=config["bias"])
        self.attn_dropout = nn.Dropout(config["dropout"])
        self.resid_dropout = nn.Dropout(config["dropout"])
        self.n_head = config["n_head"]
        self.n_embd = config["n_embd"]
        self.dropout = config["dropout"]
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config["block_size"], config["block_size"]))
                                        .view(1, 1, config["block_size"], config["block_size"]))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C//self.n_head).transpose(1,2)
        q = q.view(B, T, self.n_head, C//self.n_head).transpose(1,2)
        v = v.view(B, T, self.n_head, C//self.n_head).transpose(1,2)

        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2,-1)) * (1.0/math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config["n_embd"], 4*config["n_embd"], bias=config["bias"])
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4*config["n_embd"], config["n_embd"], bias=config["bias"])
        self.get_batch(split, config, data_root):
    if not hasattr(get_batch, "train_data"):
        train_data = np.fromfile(os.path.join(data_root, "train.bin"), dtype=np.uint16)
        val_data   = np.fromfile(os.path.join(data_root, "val.bin"),   dtype=np.uint16)
        get_batch.train_data = torch.from_numpy(train_data.astype(np.int64))
        get_batch.val_data   = torch.from_numpy(val_data.astype(np.int64))
    data = get_batch.train_data if split == 'train' else get_batch.val_data
    ix = torch.randint(len(data) - config["block_size"], (config["batch_size"],))
    x = torch.stack([data[i:i+config["block_size"]] for i in ix])
    y = torch.stack([data[i+1:i+config["block_size"]+1] for i in ix])
    return x.to(config["device"]), y.to(config["device"])

# ---------- 4. Training ----------
@torch.no_grad()
def estimate_loss(model, config):
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(config["eval_iters"])
        for k in range(config["eval_iters"]):
            X, Y = get_batch(split, config, config["data_dir"])
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def get_lr(it, config):
    if it < config["warmup_iters"]:
        return config["learning_rate"] * it / config["warmup_iters"]
    if it > config["lr_decay_iters"]:
        return config["min_lr"]
    decay_ratio = (it - config["warmup_iters"]) / (config["lr_decay_iters"] - config["warmup_iters"])
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return config["min_lr"] + coeff * (config["learning_rate"] - config["min_lr"])

def train():
    config = default_config()
    os.makedirs(config["out_dir"], exist_ok=True)
    torch.manual_seed(1337)

    meta_path = os.path.join(config["data_dir"], "meta.pkl")
    if os.path.exists(meta_path):
        vocab_size = pickle.load(open(meta_path, "rb"))["vocab_size"]
    else:
        vocab_size = 50257  # GPT-2 default
    config["vocab_size"] = vocab_size

    model = GPT(config).to(config["device"])
    print(f"Model has {model.get_num_params()/1e6:.2f} M parameters")

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=config["learning_rate"],
                                  betas=(config["beta1"], config["beta2"]),
                                  weight_decay=config["weight_decay"])

    for iter in tqdm(range(config["max_iters"])):
        lr = get_lr(iter, config)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        if iter % config["eval_interval"] == 0:
            losses = estimate_loss(model, config)
            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        x, y = get_batch('train', config, config["data_dir"])
        _, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if config["grad_clip"] != 0.0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
        optimizer.step()

        if iter % config["save_interval"] == 0 or iter == config["max_iters"] - 1:
            checkpoint = {
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'model_args': config,
                'iter_num': iter,
                'best_val_loss': losses['val'],
            }
            torch.save(checkpoint, os.path.join(config["out_dir"], 'ckpt.pt'))

# ---------- 5. Sampling ----------
def sample():
    config = default_config()
    checkpoint = torch.load(os.path.join(config["out_dir"], 'ckpt.pt'), map_location=config["device"])
    model = GPT(checkpoint['model_args']).to(config["device"])
    model.load_state_dict(checkpoint['model'])
    model.eval()

    start = "Bugün hava çok güzeldi ve"
    start_ids = tiktoken.get_encoding("gpt2").encode(start)
    x = torch.tensor(start_ids, dtype=torch.long, device=config["device"])[None, ...]

    max_new_tokens = 300
    temperature = 0.8
    top_k = 40
    for _ in range(max_new_tokens):
        x_cond = x if x.size(1) <= config["block_size"] else x[:, -config["block_size"]:]
        logits, _ = model(x_cond)
        logits = logits[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        x = torch.cat((x, ix), dim=1)
    out = tiktoken.get_encoding("gpt2").decode(x[0].tolist())
    print(out)

# ---------- 6. CLI ----------
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("mode", choices=["train", "sample"])
    args = parser.parse_args()
    if args.mode == "train":
        train()
    elif args.mode == "sample":
        sample()
f.dropout = nn.Dropout(config["dropout"])
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config["n_embd"], bias=config["bias"])
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config["n_embd"], bias=config["bias"])
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config["vocab_size"], config["n_embd"]),
            wpe = nn.Embedding(config["block_size"], config["n_embd"]),
            drop = nn.Dropout(config["dropout"]),
            h = nn.ModuleList([Block(config) for _ in range(config["n_layer"])]),
            ln_f = LayerNorm(config["n_embd"], bias=config["bias"]),
        ))
        self.lm_head = nn.Linear(config["n_embd"], config["vocab_size"], bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def get_num_params(self, non_embedding=True):
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config["block_size"], f"Cannot forward sequence of length {t}, block size is only {self.config['block_size']}"
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    def crop_block_size(self, block_size):
        assert block_size <= self.config["block_size"]
        self.config["block_size"] = block_size
        self.transformer.wpe.weight = nn.Parameter(self.transformer.wpe.weight[:block_size])
        self.register_buffer("bias", self.bias[:,:,:block_size,:block_size])



Overwriting /content/train_custom.py
